# HockeyVision – binární model

Tento notebook slouží pro natrénování binárního modelu, který rozpozná:

- hokej
- neni_hokej



In [1]:
import os
import shutil
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
SOURCE_DATASET_PATH = "/content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset"
BINARY_DATASET_PATH = "/content/binary_dataset_ready"

In [4]:
if os.path.exists(BINARY_DATASET_PATH):
    shutil.rmtree(BINARY_DATASET_PATH)

os.makedirs(os.path.join(BINARY_DATASET_PATH, "hokej"), exist_ok=True)
os.makedirs(os.path.join(BINARY_DATASET_PATH, "neni_hokej"), exist_ok=True)

In [5]:
import os

print("Existuje cesta:", os.path.exists(SOURCE_DATASET_PATH))
print("Obsah:", os.listdir(SOURCE_DATASET_PATH))

for item in os.listdir(SOURCE_DATASET_PATH):
    full_path = os.path.join(SOURCE_DATASET_PATH, item)
    print(item, "->", full_path, "| existuje:", os.path.exists(full_path))

Existuje cesta: True
Obsah: ['neni_hokej', 'klidova_situace', 'vhazovani', 'jizda_s_pukem', 'souboj_u_mantinelu', 'zakrok_brankare']
neni_hokej -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/neni_hokej | existuje: True
klidova_situace -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/klidova_situace | existuje: True
vhazovani -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/vhazovani | existuje: True
jizda_s_pukem -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/jizda_s_pukem | existuje: True
souboj_u_mantinelu -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/souboj_u_mantinelu | existuje: True
zakrok_brankare -> /content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset/zakrok_brankare | existuje: True


In [6]:
print(SOURCE_DATASET_PATH)

/content/drive/MyDrive/Colab Notebooks/HockeyVision/dataset/dataset


In [7]:
situation_folders = [
    "vhazovani",
    "souboj_u_mantinelu",
    "jizda_s_pukem",
    "zakrok_brankare",
    "klidova_situace"
]

for folder in situation_folders:
    src_folder = os.path.join(SOURCE_DATASET_PATH, folder)
    dst_folder = os.path.join(BINARY_DATASET_PATH, "hokej")

    for filename in os.listdir(src_folder):
        src_file = os.path.join(src_folder, filename)
        dst_file = os.path.join(dst_folder, f"{folder}_{filename}")
        shutil.copy2(src_file, dst_file)

neni_hokej_src = os.path.join(SOURCE_DATASET_PATH, "neni_hokej")
neni_hokej_dst = os.path.join(BINARY_DATASET_PATH, "neni_hokej")

for filename in os.listdir(neni_hokej_src):
    src_file = os.path.join(neni_hokej_src, filename)
    dst_file = os.path.join(neni_hokej_dst, filename)
    shutil.copy2(src_file, dst_file)

print("Binární dataset připraven.")
print("Třídy:", os.listdir(BINARY_DATASET_PATH))
print("Počet hokej:", len(os.listdir(os.path.join(BINARY_DATASET_PATH, "hokej"))))
print("Počet neni_hokej:", len(os.listdir(os.path.join(BINARY_DATASET_PATH, "neni_hokej"))))

Binární dataset připraven.
Třídy: ['hokej', 'neni_hokej']
Počet hokej: 731
Počet neni_hokej: 500


In [8]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42
EPOCHS = 10
DATASET_PATH = BINARY_DATASET_PATH

In [9]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary"
)

print("Třídy:", train_ds.class_names)

Found 1231 files belonging to 2 classes.
Using 985 files for training.
Found 1231 files belonging to 2 classes.
Using 246 files for validation.
Třídy: ['hokej', 'neni_hokej']


In [10]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

In [11]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.05),
    layers.RandomZoom(0.1),
])

In [12]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

base_model.trainable = False

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [13]:
inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = keras.Model(inputs, outputs)

In [14]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [15]:
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    keras.callbacks.ModelCheckpoint(
        "binary_hockey_model.keras",
        monitor="val_loss",
        save_best_only=True
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

Epoch 1/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 191s 6s/step - accuracy: 0.8487 - loss: 0.3576 - val_accuracy: 0.9553 - val_loss: 0.1444
Epoch 2/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 245s 7s/step - accuracy: 0.9685 - loss: 0.1199 - val_accuracy: 0.9756 - val_loss: 0.0997
Epoch 3/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 188s 6s/step - accuracy: 0.9746 - loss: 0.0838 - val_accuracy: 0.9756 - val_loss: 0.0865
Epoch 4/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 190s 6s/step - accuracy: 0.9756 - loss: 0.0807 - val_accuracy: 0.9756 - val_loss: 0.0767
Epoch 5/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 218s 6s/step - accuracy: 0.9827 - loss: 0.0519 - val_accuracy: 0.9837 - val_loss: 0.0671
Epoch 6/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 185s 6s/step - accuracy: 0.9807 - loss: 0.0595 - val_accuracy: 0.9878 - val_loss: 0.0608
Epoch 7/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 177s 6s/step - accuracy: 0.9838 - loss: 0.0522 - val_accuracy: 0.9919 - val_loss: 0.0535
Epoch 8/10
31/31 ━━━━━━━━━━━━━━━━━━━━ 180s 6s/step - accuracy: 0.9878 - loss: 0.0458 - val_accuracy: 0.9878 - v

In [16]:
model.save("binary_hockey_model.keras")
print("Model uložen jako binary_hockey_model.keras")

Model uložen jako binary_hockey_model.keras


In [17]:
from google.colab import files
files.download("binary_hockey_model.keras")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>